# Data Preprocessing
This notebook cleans the Global Superstore dataset and exports a cleaned CSV.

In [10]:
import pandas as pd
import numpy as np

RAW_PATH = '../data/raw/superstore.csv'
OUTPUT_PATH = '../data/cleaned/superstore_cleaned.csv'

df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()


(51290, 27)


,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,CA-2011-146969,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


## Inspect dataset

In [11]:
df.info()
df.describe(include='all').T


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        51290 non-null  object 
 1   City            51290 non-null  object 
 2   Country         51290 non-null  object 
 3   Customer.ID     51290 non-null  object 
 4   Customer.Name   51290 non-null  object 
 5   Discount        51290 non-null  float64
 6   Market          51290 non-null  object 
 7   记录数             51290 non-null  int64  
 8   Order.Date      51290 non-null  object 
 9   Order.ID        51290 non-null  object 
 10  Order.Priority  51290 non-null  object 
 11  Product.ID      51290 non-null  object 
 12  Product.Name    51290 non-null  object 
 13  Profit          51290 non-null  float64
 14  Quantity        51290 non-null  int64  
 15  Region          51290 non-null  object 
 16  Row.ID          51290 non-null  int64  
 17  Sales           51290 non-null 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Category,51290,3,Office Supplies,31273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,51290,3636,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,51290,147,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer.ID,51290,4873,JG-158051,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer.Name,51290,795,Muhammed Yedwab,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Discount,51290.0,NaN,NaN,NaN,0.142908,0.21228,0.0,0.0,0.0,0.2,0.85
Market,51290,7,APAC,11002,NaN,NaN,NaN,NaN,NaN,NaN,NaN
记录数,51290.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
Order.Date,51290,1430,2014-06-18 00:00:00.000,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order.ID,51290,25035,CA-2014-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing values

In [12]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing>0]


Series([], dtype: int64)

## Remove duplicate rows

In [13]:
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)


Duplicates: 0
(51290, 27)


Drop unnecessary columns

In [14]:

columns_to_drop = [
    "记录数",      
]

df = df.drop(columns=columns_to_drop)

## Rename columns

In [15]:
df.columns = (
    df.columns.str.strip()
              .str.replace('.', '_', regex=False)
              .str.replace(' ', '_', regex=False)
              .str.replace('/', '_', regex=False)
)
df.columns.tolist()


['Category',
 'City',
 'Country',
 'Customer_ID',
 'Customer_Name',
 'Discount',
 'Market',
 'Order_Date',
 'Order_ID',
 'Order_Priority',
 'Product_ID',
 'Product_Name',
 'Profit',
 'Quantity',
 'Region',
 'Row_ID',
 'Sales',
 'Segment',
 'Ship_Date',
 'Ship_Mode',
 'Shipping_Cost',
 'State',
 'Sub_Category',
 'Year',
 'Market2',
 'weeknum']

## Convert dates

In [16]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])


## Feature engineering

In [17]:
df['Shipping_Days']=(df['Ship_Date']-df['Order_Date']).dt.days
df['Order_Year']=df['Order_Date'].dt.year
df['Order_Month']=df['Order_Date'].dt.month
df['Order_Month_Name']=df['Order_Date'].dt.month_name()
df['Order_Quarter']=df['Order_Date'].dt.quarter
df['Order_Weekday']=df['Order_Date'].dt.day_name()
df['Profit_Margin_%']=np.where(df['Sales']!=0,(df['Profit']/df['Sales'])*100,0)


## Check negative values

In [18]:
df[['Sales','Profit','Quantity','Discount','Shipping_Cost']].describe()


,Sales,Profit,Quantity,Discount,Shipping_Cost
count,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000
mean,246.498440,28.610982,3.476545,0.142908,26.375818
std,487.567175,174.340972,2.278766,0.212280,57.296810
min,0.000000,-6599.978000,1.000000,0.000000,0.002000
25%,31.000000,0.000000,2.000000,0.000000,2.610000
50%,85.000000,9.240000,3.000000,0.000000,7.790000
75%,251.000000,36.810000,5.000000,0.200000,24.450000
max,22638.000000,8399.976000,14.000000,0.850000,933.570000


## Export cleaned dataset

In [19]:
import os
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv(OUTPUT_PATH,index=False)
print(f'Saved to {OUTPUT_PATH}')
df.head()


Saved to ../data/cleaned/superstore_cleaned.csv


,Category,City,Country,Customer_ID,Customer_Name,Discount,Market,Order_Date,Order_ID,Order_Priority,...,Year,Market2,weeknum,Shipping_Days,Order_Year,Order_Month,Order_Month_Name,Order_Quarter,Order_Weekday,Profit_Margin_%
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,2011-01-07,CA-2011-130813,High,...,2011,North America,2,2,2011,1,January,1,Friday,49.111579
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,2011-01-21,CA-2011-148614,Medium,...,2011,North America,4,5,2011,1,January,1,Friday,48.909474
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,2011-08-05,CA-2011-118962,Medium,...,2011,North America,32,4,2011,8,August,3,Friday,46.865714
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,2011-08-05,CA-2011-118962,Medium,...,2011,North America,32,4,2011,8,August,3,Friday,47.982703
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,2011-09-29,CA-2011-146969,High,...,2011,North America,40,4,2011,9,September,3,Thursday,51.840000
